In [ ]:
from vllm import LLM, SamplingParams
from typing import List, Optional, Dict, Union
import logging
from datetime import datetime
import json
import torch

class DeepSeekVLLM:
    def __init__(
        self,
        model_name: str = "deepseek-ai/deepseek-coder-1.3b-base",
        temperature: float = 0.7,
        max_tokens: int = 2048,
        top_p: float = 0.95,
        tensor_parallel_size: int = 1,
        max_model_len: int = 8192,  # Reduced from default
        verbose: bool = False
    ):
        """
        Initialize DeepSeek with vLLM
        
        Args:
            model_name: Name or path of the model
            temperature: Sampling temperature
            max_tokens: Maximum number of tokens to generate
            top_p: Top-p sampling parameter
            tensor_parallel_size: Number of GPUs for tensor parallelism
            max_model_len: Maximum sequence length
            verbose: Enable verbose logging
        """
        self.verbose = verbose
        self._setup_logging()
        
        self.logger.info(f"Initializing DeepSeek with vLLM using model: {model_name}")
        
        # Initialize vLLM model with adjusted parameters
        self.model = LLM(
            model=model_name,
            tensor_parallel_size=tensor_parallel_size,
            trust_remote_code=True,
            dtype="float16",
            gpu_memory_utilization=0.95,  # Increased memory utilization
            max_model_len=max_model_len,  # Set maximum sequence length
            quantization=None,
            enforce_eager=True  # Disable CUDA graph to reduce memory usage
        )
        
        # Set default sampling parameters
        self.sampling_params = SamplingParams(
            temperature=temperature,
            max_tokens=max_tokens,
            top_p=top_p,
            presence_penalty=0.0,
            frequency_penalty=0.0
        )
        
        self.conversation_history = []
        
    def _setup_logging(self):
        """Setup logging configuration"""
        logging.basicConfig(
            level=logging.INFO if self.verbose else logging.WARNING,
            format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
        )
        self.logger = logging.getLogger(__name__)
        
    def generate(
        self,
        prompt: str,
        sampling_params: Optional[SamplingParams] = None
    ) -> str:
        """
        Generate response for a single prompt
        
        Args:
            prompt: Input prompt
            sampling_params: Optional custom sampling parameters
            
        Returns:
            Generated response
        """
        try:
            params = sampling_params or self.sampling_params
            outputs = self.model.generate([prompt], params)
            response = outputs[0].outputs[0].text
            
            # Store in conversation history
            self.conversation_history.append({
                "prompt": prompt,
                "response": response,
                "timestamp": datetime.now().isoformat()
            })
            
            return response
            
        except Exception as e:
            self.logger.error(f"Error during generation: {str(e)}")
            raise
            
    def batch_generate(
        self,
        prompts: List[str],
        sampling_params: Optional[SamplingParams] = None
    ) -> List[str]:
        """
        Generate responses for multiple prompts in parallel
        
        Args:
            prompts: List of input prompts
            sampling_params: Optional custom sampling parameters
            
        Returns:
            List of generated responses
        """
        try:
            params = sampling_params or self.sampling_params
            outputs = self.model.generate(prompts, params)
            responses = [output.outputs[0].text for output in outputs]
            
            # Store in conversation history
            for prompt, response in zip(prompts, responses):
                self.conversation_history.append({
                    "prompt": prompt,
                    "response": response,
                    "timestamp": datetime.now().isoformat()
                })
                
            return responses
            
        except Exception as e:
            self.logger.error(f"Error during batch generation: {str(e)}")
            raise



In [ ]:
def clear_gpu_memory():
    """Clear GPU memory and cache"""
    # Empty CUDA cache
    torch.cuda.empty_cache()
    
    # Force garbage collection
    gc.collect()
    
    # Reset peak memory stats
    torch.cuda.reset_peak_memory_stats()
    
    # Optional: print memory status
    if torch.cuda.is_available():
        print(f"GPU Memory Summary after clearing:")
        print(f"Allocated: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
        print(f"Cached: {torch.cuda.memory_reserved() / 1024**2:.2f} MB")

import torch
import gc
import psutil

def get_gpu_memory_usage():
    """Detailed GPU memory analysis"""
    print("\n=== GPU Memory Usage Analysis ===")
    
    # Get GPU properties
    if torch.cuda.is_available():
        gpu_props = torch.cuda.get_device_properties(0)
        print(f"\nGPU Device: {gpu_props.name}")
        print(f"Total GPU Memory: {gpu_props.total_memory / 1024**2:.2f} MB")
        
        # Current memory usage
        print("\nCurrent Memory Status:")
        print(f"Allocated: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
        print(f"Cached: {torch.cuda.memory_reserved() / 1024**2:.2f} MB")
        
        # Get all tensors in memory
        print("\nTensors in Memory:")
        for obj in gc.get_objects():
            try:
                if torch.is_tensor(obj) or (hasattr(obj, 'data') and torch.is_tensor(obj.data)):
                    print(type(obj), obj.size(), obj.dtype, obj.device)
            except:
                pass
        
        # System memory info
        print("\nSystem Memory:")
        vm = psutil.virtual_memory()
        print(f"Total: {vm.total / 1024**3:.2f} GB")
        print(f"Available: {vm.available / 1024**3:.2f} GB")
        print(f"Used: {vm.used / 1024**3:.2f} GB")
        print(f"Percentage: {vm.percent}%")

def force_clear_gpu_memory():
    """Aggressive GPU memory clearing"""
    print("\n=== Clearing GPU Memory ===")
    
    # Delete all variables
    for obj in gc.get_objects():
        try:
            if torch.is_tensor(obj) or (hasattr(obj, 'data') and torch.is_tensor(obj.data)):
                del obj
        except:
            pass
    
    # Force garbage collection
    gc.collect()
    
    # Empty CUDA cache
    torch.cuda.empty_cache()
    
    # Reset peak memory stats
    torch.cuda.reset_peak_memory_stats()
    
    # Optional: Reset device
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.empty_cache()
        
        # Print final status
        print("\nMemory Status After Clearing:")
        print(f"Allocated: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
        print(f"Cached: {torch.cuda.memory_reserved() / 1024**2:.2f} MB")



In [ ]:
# Example usage
if __name__ == "__main__":
    get_gpu_memory_usage()
    force_clear_gpu_memory()
    # Print initial GPU memory status
    print("Initial GPU Memory Status:")
    
    # Initialize DeepSeek with vLLM
    deepseek = DeepSeekVLLM(
        max_model_len=8192,  # Explicitly set lower max length
        verbose=True
    )
    
    # Test with a simple prompt
    prompt = "correct spelling of valaubel"
    
    try:
        response = deepseek.generate(prompt)
        print(f"\nPrompt: {prompt}")
        print(f"Response: {response}")
        clear_gpu_memory()
    except Exception as e:
        print(f"Error during generation: {str(e)}")
        clear_gpu_memory()